In [3]:
import sqlite3

### Ettevalmistus

In [4]:
# verbimustrite andmebaas
pattern_db = "../example_data/verb_patterns_actors.db"

# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# Siia salvestuvad loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"

# state of the verb (isikumäärus)
# alati: alati isikumäärus etc
STAT = 'alati' #'mitte_kunagi'

In [5]:
# verbimustrite andmebaas, hetkel kasutan sealt ainult tabelit verb_patterns_len1
con = sqlite3.connect(pattern_db)
cur = con.cursor()

# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{transaction_db}" as trans ')

# uue andmebaasi lisamine. 
cur.execute(f'ATTACH DATABASE "{vp_data_db}" AS vp')

In [6]:
# tabelisse patterns_len1 lisatakse uus veerg phrase_nr, mille kõigi ridade väärtuseks on 1
cur.execute("""
ALTER TABLE patterns_actors_len1_{stat}
ADD COLUMN
    phrase_nr INTEGER
""".format(stat=STAT))

cur.execute("""
UPDATE patterns_actors_len1_{stat}
SET phrase_nr = 1
""".format(stat=STAT))

con.commit()

### I tabel patterns

Veerud:

    pat_id - (mustri ID tabelis patterns_len1)
    pattern - (algne muster sõnena; NB! HETKEL ON SELLE VEERU READ MINGIL PÕHJUSEL NIHKES, SEETÕTTU PALUN HETKEL SEDA IGNOREERIDA, PARANDAN HILJEM)
    verb_word - (mustri (pea)verb)
    verb_compound - (pikema verbiühendi ülejäänud osad)
    phrase_nr - (fraasi number; kuna hetkel on vaatluse all ainult tabelist patterns_len1 pärit mustrid, on kõigil fraasidel number 1)
    phrase_case - (fraasi põhiliikme (pärast verbi) kääne; hiljem vaatame ilmselt vaid fraase, kus selles käändes on obliikva, kuid praeguseks pole seda tingimust veel sisse pandud)
    adp - (kaassõna)
    inf_verb - (infiniitverb)
    
Vajalik info saadakse tabelitest patterns_len1 ja transaction_head.
Hetkel on märgitud limiidiks 200, sest protseduur on aeganõudev. See tähendab, et tabelis patterns on tabelist patterns_len1 esimesed 200 mustrit, millel on lisaks põhiverbile kuni üks verbiühendi osa ning millel ei ole tabelis patterns_len1 *other*-kategooriasse kuuluvat mustriliiget.
Verbiühendeid, millel on lisaks põhiverbile rohkem, kui üks osa, on transaktsioonide andmebaasis võrdlemisi vähe.
*Other*-kategooria lahendamine on natuke keerukas (sidesõnad, sõnad nagu 'millal', 'kuidas' jne).

In [7]:
%%time

cur.execute("""
DROP TABLE IF EXISTS vp.patterns_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE vp.patterns_actors_{stat1} AS
SELECT DISTINCT
    pat.ID AS pat_id,
    pat.word || ' ' || pat.government AS pattern,
    pat.verb_word AS verb_word,
    pat.compound_prt1 AS verb_compound,
    pat.phrase_nr AS phrase_nr,
    pat.w_case AS phrase_case,
    pat.adp AS adp,
    pat.verb AS inf_verb,
    pat.deprel as pat_deprel
FROM 
    patterns_actors_len1_{stat} as pat
INNER JOIN 
    trans.transaction_head
ON 
    pat.verb_word = trans.transaction_head.verb
WHERE 
    pat.compound_prt1 = trans.transaction_head.verb_compound
AND 
    pat.compound_prt2 = ''
AND 
    pat.compound_prt3 = ''
AND 
    pat.other = ''
""".format(stat1=STAT, stat=STAT))



CPU times: user 1.87 ms, sys: 3.45 ms, total: 5.32 ms
Wall time: 10.5 ms


cur.execute("""
CREATE INDEX vp.pat_id_idx ON patterns(pat_id)
"""
)

cur.execute("""
CREATE INDEX vp.phrase_case_idx ON patterns(phrase_case)
"""
)

cur.execute("""
CREATE INDEX vp.adp_idx ON patterns(adp)
"""
)

cur.execute("""
CREATE INDEX vp.inf_verb_idx ON patterns(inf_verb)
"""
)

cur.execute("""
CREATE INDEX vp.verb_word_idx ON patterns(verb_word)
"""
)

cur.execute("""
CREATE INDEX vp.verb_compound_idx ON patterns(verb_compound)
"""
)

cur.execute("""
CREATE INDEX vp.phrase_nr_idx ON patterns(phrase_nr)
"""
)

### Abitabel asjade kättesaamiseks

Mitte-elegantne viis saada kätte kõik **head_id**-d, millele vastavates fraasides on esindatud kõik vaadeldavate mustrite osised (sobiv kääne (kui on), kaassõna (kui on), infiniitverb (kui on)). Saab kasutada ülejäänud tabelite koostamiseks. Ilmselt on võimalik teha tegelikult ära ka JOIN-ide abil. Limiidiks on 500, sest tegemist on taaskord üsna aeganõudva protsessiga.

In [8]:
%%time

cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step1
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step2
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step3
""")

cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step1 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
(
    SELECT pat.pat_id as pat_id,
        tr_head.id as head_id,
        pat.phrase_case as phrase_case,
        pat.adp as adp,
        pat.inf_verb as inf_verb,
        pat.phrase_nr as phrase_nr
    FROM 
        vp.patterns_actors_{stat} as pat
    INNER JOIN 
        trans.transaction_head as tr_head
    ON
        pat.verb_word=tr_head.verb
    WHERE
        pat.verb_compound=tr_head.verb_compound
) as pat_tr_joined
INNER JOIN 
    trans.`transaction` as tr
ON
    pat_tr_joined.head_id=tr.head_id
WHERE 
    pat_tr_joined.phrase_case = '' OR instr(tr.feats, pat_tr_joined.phrase_case) > 0
""".format(stat=STAT))


CPU times: user 2.4 ms, sys: 2.82 ms, total: 5.23 ms
Wall time: 8.04 ms


In [9]:
%%time

cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step2 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb,
    phrase_nr
FROM 
    vp.verb_phrase_matches_step1 as step1
INNER JOIN 
    trans.`transaction` as tr
ON 
    step1.head_id=tr.head_id
WHERE 
    step1.adp = '' OR (tr.form = step1.adp AND tr.deprel = 'case')
""")


CPU times: user 758 µs, sys: 3.26 ms, total: 4.02 ms
Wall time: 7.79 ms


In [10]:
%%time

cur.execute("""
CREATE TABLE vp.verb_phrase_matches_step3 AS
SELECT pat_id,
    tr.head_id,
    phrase_case,
    adp,
    inf_verb, 
    phrase_nr
FROM 
    vp.verb_phrase_matches_step2 as step2
INNER JOIN
    trans.`transaction` as tr
ON 
    step2.head_id=tr.head_id
WHERE 
    step2.inf_verb = '' 
    OR (tr.form = step2.inf_verb AND (instr(tr.feats, 'inf') > 0 
    OR instr(tr.feats, 'sup') > 0))
""")


cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step1
""")
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step2
""")

CPU times: user 2.24 ms, sys: 4.05 ms, total: 6.29 ms
Wall time: 17.3 ms


### II tabel patterns_meta

Veerud:

    pat_id - mustri ID
    phrase_count - mustrile vastavate fraaside (limiteeritud) hulk transaktsioonide andmebaasis
    
Asjade kättesaamiseks kasutan abitabelit, sealt leian esiteks unikaalsed read, mis seejärel kokku loetakse. Tuleb endiselt silmas pidada, et vastete hulk transaktsioonide andmebaasist on hetkel piiratud.

In [11]:
%%time

cur.execute("""
DROP TABLE IF EXISTS vp.patterns_meta_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE vp.patterns_meta_actors_{stat} (
    pat_id INTEGER,
    phrase_count INTEGER
)
""".format(stat=STAT))

cur.execute("""
INSERT INTO vp.patterns_meta_actors_{stat}(
    pat_id,
    phrase_count
)
SELECT 
    pat_id,
    count(*) AS phrase_count
FROM
(
    SELECT DISTINCT
        pat_id, 
        head_id
    FROM
        vp.verb_phrase_matches_step3
) AS tbl
GROUP BY
    tbl.pat_id
ORDER BY
    phrase_count DESC
""".format(stat=STAT))

cur.execute("""
CREATE INDEX vp.meta_pat_id_idx_{stat1} ON patterns_meta_actors_{stat}(pat_id)
""".format(stat1 = STAT, stat=STAT))

con.commit()

CPU times: user 2.94 ms, sys: 1.04 ms, total: 3.98 ms
Wall time: 10.8 ms


### III tabel verb_phrase_matches

Veerud:

    pat_id - mustri ID
    head_id - verbi ID transaktsioonide andmebaasis
    phrase_nr - fraasi nr
    
Asjade kättesaamiseks kasutan abitabelit, sealt leian unikaalsed read. Tuleb endiselt silmas pidada, et vastete hulk transaktsioonide andmebaasist on hetkel piiratud.
Kui tulevikus võtta juurde pikemad mustrid, mis sisaldavad rohkem, kui ühte fraasi, siis saab esimene fraas olema eristatud tähistatud numbriga 1 ning teine numbriga 2.

In [14]:
%%time

cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE vp.verb_phrase_matches_actors_{stat} (
    pat_id INTEGER,
    head_id INTEGER,
    phrase_nr INTEGER
)
""".format(stat=STAT))

cur.execute("""
INSERT INTO vp.verb_phrase_matches_actors_{stat}(
    pat_id,
    head_id,
    phrase_nr
)
SELECT DISTINCT
    pat_id,
    head_id,
    phrase_nr
FROM
    vp.verb_phrase_matches_step3
""".format(stat=STAT))

cur.execute("""
CREATE INDEX vp.match_pat_id_idx_{stat1} ON verb_phrase_matches_actors_{stat}(pat_id)
""".format(stat1 = STAT, stat=STAT))

cur.execute("""
CREATE INDEX vp.match_head_id_idx_{stat1} ON verb_phrase_matches_actors_{stat}(head_id)
""".format(stat1 = STAT, stat=STAT))

cur.execute("""
CREATE INDEX vp.match_phrase_nr_idx_{stat1} ON verb_phrase_matches_actors_{stat}(phrase_nr)
""".format(stat1 = STAT, stat=STAT))


con.commit()

CPU times: user 4.41 ms, sys: 187 µs, total: 4.6 ms
Wall time: 10.1 ms


### IV tabel verb_matches

Veerud:

    pat_id - mustri ID
    head_id - verbi ID transaktsioonide andmebaasis
    
Asjade kättesaamiseks kasutan tabelit patterns, et saada kätte (piiratud hulga) mustrite verbid ning transaktsioonide tabelit transaction_head, et saada kätte viited kõigile lausetele, kus verbid esinevad. Ülejäänud mustrit pole selle tabeli puhul arvesse võetud.

In [15]:
%%time

# tabel verb_matches
cur.execute("""
DROP TABLE IF EXISTS vp.verb_matches_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE vp.verb_matches_actors_{stat} (
    pat_id INTEGER,
    head_id INTEGER
)
""".format(stat=STAT))

cur.execute("""
INSERT INTO vp.verb_matches_actors_{stat}(
    pat_id,
    head_id
)
SELECT DISTINCT
    pat_id,
    head_id
FROM
(
    SELECT pat_id, tr_head.id as head_id
    FROM
        patterns_actors_{stat2} as pat
    INNER JOIN
        trans.transaction_head as tr_head
    ON
        pat.verb_word = tr_head.verb
    WHERE pat.verb_compound = tr_head.verb_compound
)
""".format(stat=STAT, stat2=STAT))

cur.execute("""
CREATE INDEX vp.v_match_pat_id_idx_{stat1} ON verb_matches_actors_{stat}(pat_id)
""".format(stat1 = STAT, stat=STAT))

cur.execute("""
CREATE INDEX vp.v_match_head_id_idx_{stat1} ON verb_matches_actors_{stat}(head_id)
""".format(stat1 = STAT, stat=STAT))

con.commit()

CPU times: user 4.74 ms, sys: 1.14 ms, total: 5.87 ms
Wall time: 14.7 ms


In [16]:
# soovi korral saab kustutada abitabeli, aga SQLITE-s see tegevus mäluruumi ei vabasta
cur.execute("""
DROP TABLE IF EXISTS vp.verb_phrase_matches_step3
""")

In [17]:
# ühenduse sulgemine
con.close()